In [1]:
import os
from tqdm import tqdm
from dotenv import load_dotenv


load_dotenv()
MY_HUGGIEFACE_TOKEN = os.getenv("MY_HUGGIEFACE_TOKEN")
MY_GEMINI_KEY = os.getenv("GEMINI_API_KEY")

OUTPUT_DIR = "./outputs/"

try:
    os.mkdir(OUTPUT_DIR)
    print("Directory 'new_directory' created.")
except FileExistsError:
    print("Directory 'new_directory' already exists.")
except FileNotFoundError:
    print("Parent directory does not exist.")


Directory 'new_directory' already exists.


In [3]:
!huggingface-cli download openthaigpt/openthaigpt-1.0.0-beta-7b-chat --resume-download

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 4 files:   0%|                                   | 0/4 [00:00<?, ?it/s]/Users/thacharo/Documents/บพค/feedMe-feedback/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Fetching 4 files: 100%|█████████████████████████| 4/4 [00:00<00:00, 3502.55it/s]
/Users/thacharo/.cache/huggingface/hub/models--openthaigpt--openthaigpt-1.0.0-beta-7b-chat/snapshots/acb6ed3462f30bcd43202780101c2674799a1c14


In [188]:
import pandas as pd

df = pd.read_csv("./test_set_no_label.csv")

In [189]:
# test_df = df.head(20).copy()
test_df = df
test_df['text']

0      อาหารไม่อร่อยอย่างที่อวยกันเลย  อาหารไม่สด ซาซ...
1      เมื่อวาน4มค63โทรจองตอนประมาณ11:55 ไปกินตอนเที่...
2      The food was decent, but the overall experienc...
3      เจอยางในอาหาร ส่งให้พนักงานดู พนักงานเป็นต่างด...
4      I was quite disappointed with the marinated cr...
                             ...                        
495    ลง BTS อโศก เดินมาไม่ไกล ร้านอยู่ชั้น 4 ตึก Ti...
496    ไปกินช่วง4-5โมง เงียบดี มี2-3โต๊ะเอง บริการดี ...
497    ร้านชาบูนางใน สาขานี้อร่อยมากกุ้งแกะให้แล้ว\nล...
498    อาหารอร่อย สะอาด\nคุ้มราคา\nบุฟเฟ่ต์ 389 น้ำชา...
499    อร่อย ราคา300กว่าบาทต่อคน ไม่จำกัดเวลา ที่ทำสำ...
Name: text, Length: 500, dtype: object

# STEP 1: Sentimental Analysis

In [190]:
from transformers import pipeline

# Initialize the pipeline for sentiment analysis
# It will automatically download the model files shown in your screenshot
# Token indices sequence length is longer than the specified maximum sequence length for this model (603 > 510). Running this sequence through the model will result in indexing errors
classifier = pipeline("sentiment-analysis", model="SiemonCha/thai-sentiment-phayabert")

Device set to use mps:0


In [7]:
def predict_long_text(text, classifier, max_len=500):
	# 1. Split text into chunks of 500 characters (approx tokens)
	# Thai doesn't use spaces, so character splitting is safer than word splitting
	chunks = [text[i:i+max_len] for i in range(0, len(text), max_len)]
	
	results = []
	for chunk in chunks:
		# Run classifier on this specific chunk
		res = classifier(chunk, truncation=True, max_length=512)[0]
		results.append(res)
	
	# 2. Logic to combine results
	# Strategy: If ANY chunk is 'neg' (negative), consider the whole thing negative.
	# Otherwise, take the one with the highest confidence score.
	
	# Check if any chunk is negative
	neg_results = [r for r in results if r['label'] == 'neg']
	if neg_results:
		# Return the strongest negative result
		return max(neg_results, key=lambda x: x['score'])
	
	# Otherwise return the strongest result overall
	return max(results, key=lambda x: x['score'])

In [191]:
labels = {'LABEL_0': "Positive", 'LABEL_1': "Neutral", 'LABEL_2': "Negative"}

for index, row in tqdm(test_df.iterrows()):
	text = row['text']
	result = predict_long_text(text, classifier)
	label = labels[result["label"]]
	score = result["score"]
	# print(text)
	# print(f"[Result: {label}, Score: {score}]\n")
	test_df.loc[index, 'sentiment'] = label
	test_df.loc[index, 'sentiment_score'] = score

500it [00:35, 14.00it/s]


# Create Test for sentimental

In [192]:
test_df['sentiment_test'] = None

for index, row in test_df.iterrows():
	rating = int(row['rating'])
	if 4 <= rating <= 5 :
		test_df.loc[index, 'sentiment_test'] = 'Positive'
	elif rating == 3:
		test_df.loc[index, 'sentiment_test'] = 'Neutral'
	else:
		test_df.loc[index, 'sentiment_test'] = 'Negative'

test_df

,Unnamed: 0,review_id,author,rating,text,sentiment,sentiment_score,sentiment_test
0,55,ChZDSUhNMG9nS0VJQ0FnSUNldWNubVpnEAE,Prachai Siriratikul,1,อาหารไม่อร่อยอย่างที่อวยกันเลย อาหารไม่สด ซาซ...,Negative,0.999561,Negative
1,63,ChZDSUhNMG9nS0VJQ0FnSURNdWZQalR3EAE,heyday business,1,เมื่อวาน4มค63โทรจองตอนประมาณ11:55 ไปกินตอนเที่...,Negative,0.999630,Negative
2,103,ChZDSUhNMG9nS0VJQ0FnTURvczhDMGJnEAE,A* Luo,1,"The food was decent, but the overall experienc...",Negative,0.999209,Negative
3,107,Ci9DQUlRQUNvZENodHljRjlvT25oNGJGSm9ORzlxWTIxUV...,natthartath thongcharoen,1,เจอยางในอาหาร ส่งให้พนักงานดู พนักงานเป็นต่างด...,Negative,0.999675,Negative
4,110,ChdDSUhNMG9nS0VLbVR0SkNScnF6UzNRRRAB,Shin,1,I was quite disappointed with the marinated cr...,Negative,0.999403,Negative
...,...,...,...,...,...,...,...,...
495,209,ChdDSUhNMG9nS0VJQ0FnSURubnV2czdnRRAB,Lts Spntps,5,ลง BTS อโศก เดินมาไม่ไกล ร้านอยู่ชั้น 4 ตึก Ti...,Positive,0.999467,Positive
496,210,Ci9DQUlRQUNvZENodHljRjlvT2pOWlVqTktNM05QYTBwMV...,Fern Pari,5,ไปกินช่วง4-5โมง เงียบดี มี2-3โต๊ะเอง บริการดี ...,Positive,0.997975,Positive
497,211,ChZDSUhNMG9nS0VJQ0FnSURIdGZHQWVREAE,Will PvR,5,ร้านชาบูนางใน สาขานี้อร่อยมากกุ้งแกะให้แล้ว\nล...,Positive,0.999308,Positive
498,212,ChdDSUhNMG9nS0VJQ0FnSUNfM051WGlBRRAB,tnpfar,5,อาหารอร่อย สะอาด\nคุ้มราคา\nบุฟเฟ่ต์ 389 น้ำชา...,Positive,0.999680,Positive


More manual

In [10]:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import torch

# model = AutoModelForSequenceClassification.from_pretrained("SiemonCha/thai-sentiment-phayabert")
# tokenizer = AutoTokenizer.from_pretrained("SiemonCha/thai-sentiment-phayabert")

# reviews = [
#     "ร้านนี้อร่อยมาก บริการดีสุดๆ",          # ควรได้ positive
#     "รอนานมาก อาหารก็เย็นชืด ไม่ไหวเลย",   # ควรได้ negative
#     "ร้านเปิดกี่โมงครับ",                    # ควรได้ neutral (หรือ q)
#     "รสชาติงั้นๆ เฉยๆ พอกินได้"                  # ควรได้ neutral
# ]

# for text in reviews:
# 	inputs = tokenizer(text, return_tensors="pt")
# 	outputs = model(**inputs)
# 	prediction = torch.argmax(outputs.logits, dim=-1).item()

# 	labels = {0: "positive", 1: "neutral", 2: "negative"}
# 	print(labels[prediction])  # positive

---

# STEP 2: Zero-shot-classification

multi_label=True: It runs a Sigmoid function on each label's "Entailment" vs. "Contradiction" logit.
Each label gets a score between 0 and 1 independent of the others.

In [11]:
classifier_th = pipeline(
	"zero-shot-classification",
	model="joeddav/xlm-roberta-large-xnli" # Supports Thai
)

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [193]:

candidate_labels = ["Taste", "Portion", "Price", "Quality", "Service", "Speed", "Location"]
ACCEPTABLE_SCORE = 0.7

test_df['keywords'] = None
for index, row in tqdm(test_df.iterrows()):
	text = row['text']
	label_lst = []
	result = classifier_th(text, candidate_labels, multi_label=True)
	# print(f"Text: {text}")
	for label, score in zip(result['labels'], result['scores']):
		if score > ACCEPTABLE_SCORE:
			# print(f"{label}: {score:.4f}")
			label_lst.append(label)
			test_df.loc[index, label] = 1
		else:
			test_df.loc[index, label] = 0
	test_df.at[index, "keywords"] = label_lst

500it [08:35,  1.03s/it]


In [194]:
test_df

,Unnamed: 0,review_id,author,rating,text,sentiment,sentiment_score,sentiment_test,keywords,Service,Location,Portion,Speed,Price,Taste,Quality
0,55,ChZDSUhNMG9nS0VJQ0FnSUNldWNubVpnEAE,Prachai Siriratikul,1,อาหารไม่อร่อยอย่างที่อวยกันเลย อาหารไม่สด ซาซ...,Negative,0.999561,Negative,"[Service, Location]",1.0,1.0,0.0,0.0,0.0,0.0,0.0
1,63,ChZDSUhNMG9nS0VJQ0FnSURNdWZQalR3EAE,heyday business,1,เมื่อวาน4มค63โทรจองตอนประมาณ11:55 ไปกินตอนเที่...,Negative,0.999630,Negative,"[Price, Service]",1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,103,ChZDSUhNMG9nS0VJQ0FnTURvczhDMGJnEAE,A* Luo,1,"The food was decent, but the overall experienc...",Negative,0.999209,Negative,"[Service, Price]",1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,107,Ci9DQUlRQUNvZENodHljRjlvT25oNGJGSm9ORzlxWTIxUV...,natthartath thongcharoen,1,เจอยางในอาหาร ส่งให้พนักงานดู พนักงานเป็นต่างด...,Negative,0.999675,Negative,"[Service, Location]",1.0,1.0,0.0,0.0,0.0,0.0,0.0
4,110,ChdDSUhNMG9nS0VLbVR0SkNScnF6UzNRRRAB,Shin,1,I was quite disappointed with the marinated cr...,Negative,0.999403,Negative,"[Portion, Price, Location]",0.0,1.0,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,209,ChdDSUhNMG9nS0VJQ0FnSURubnV2czdnRRAB,Lts Spntps,5,ลง BTS อโศก เดินมาไม่ไกล ร้านอยู่ชั้น 4 ตึก Ti...,Positive,0.999467,Positive,"[Price, Quality, Taste, Location]",0.0,1.0,0.0,0.0,1.0,1.0,1.0
496,210,Ci9DQUlRQUNvZENodHljRjlvT2pOWlVqTktNM05QYTBwMV...,Fern Pari,5,ไปกินช่วง4-5โมง เงียบดี มี2-3โต๊ะเอง บริการดี ...,Positive,0.997975,Positive,"[Service, Quality, Taste, Location, Price]",1.0,1.0,0.0,0.0,1.0,1.0,1.0
497,211,ChZDSUhNMG9nS0VJQ0FnSURIdGZHQWVREAE,Will PvR,5,ร้านชาบูนางใน สาขานี้อร่อยมากกุ้งแกะให้แล้ว\nล...,Positive,0.999308,Positive,"[Taste, Price, Location, Quality]",0.0,1.0,0.0,0.0,1.0,1.0,1.0
498,212,ChdDSUhNMG9nS0VJQ0FnSUNfM051WGlBRRAB,tnpfar,5,อาหารอร่อย สะอาด\nคุ้มราคา\nบุฟเฟ่ต์ 389 น้ำชา...,Positive,0.999680,Positive,"[Quality, Price, Taste]",0.0,0.0,0.0,0.0,1.0,1.0,1.0


In [196]:
mock_review_df = test_df[['review_id', 'text', 'sentiment', 'sentiment_score', 'keywords']]
mock_review_df = mock_review_df.rename(columns={
	"review_id":"id",
	"sentiment_score":"confidence",
})
mock_review_df.to_csv(f"{OUTPUT_DIR}test_set_with_model_label.csv", index=False)
mock_review_df

,id,text,sentiment,confidence,keywords
0,ChZDSUhNMG9nS0VJQ0FnSUNldWNubVpnEAE,อาหารไม่อร่อยอย่างที่อวยกันเลย อาหารไม่สด ซาซ...,Negative,0.999561,"[Service, Location]"
1,ChZDSUhNMG9nS0VJQ0FnSURNdWZQalR3EAE,เมื่อวาน4มค63โทรจองตอนประมาณ11:55 ไปกินตอนเที่...,Negative,0.999630,"[Price, Service]"
2,ChZDSUhNMG9nS0VJQ0FnTURvczhDMGJnEAE,"The food was decent, but the overall experienc...",Negative,0.999209,"[Service, Price]"
3,Ci9DQUlRQUNvZENodHljRjlvT25oNGJGSm9ORzlxWTIxUV...,เจอยางในอาหาร ส่งให้พนักงานดู พนักงานเป็นต่างด...,Negative,0.999675,"[Service, Location]"
4,ChdDSUhNMG9nS0VLbVR0SkNScnF6UzNRRRAB,I was quite disappointed with the marinated cr...,Negative,0.999403,"[Portion, Price, Location]"
...,...,...,...,...,...
495,ChdDSUhNMG9nS0VJQ0FnSURubnV2czdnRRAB,ลง BTS อโศก เดินมาไม่ไกล ร้านอยู่ชั้น 4 ตึก Ti...,Positive,0.999467,"[Price, Quality, Taste, Location]"
496,Ci9DQUlRQUNvZENodHljRjlvT2pOWlVqTktNM05QYTBwMV...,ไปกินช่วง4-5โมง เงียบดี มี2-3โต๊ะเอง บริการดี ...,Positive,0.997975,"[Service, Quality, Taste, Location, Price]"
497,ChZDSUhNMG9nS0VJQ0FnSURIdGZHQWVREAE,ร้านชาบูนางใน สาขานี้อร่อยมากกุ้งแกะให้แล้ว\nล...,Positive,0.999308,"[Taste, Price, Location, Quality]"
498,ChdDSUhNMG9nS0VJQ0FnSUNfM051WGlBRRAB,อาหารอร่อย สะอาด\nคุ้มราคา\nบุฟเฟ่ต์ 389 น้ำชา...,Positive,0.999680,"[Quality, Price, Taste]"


# STEP 3: Aggregation

In [167]:
mapping_df = test_df[['sentiment'] + candidate_labels].copy()
aggregate_df = mapping_df.groupby('sentiment').sum().transpose().copy()

In [168]:
# aggregate_df.to_json(f"{OUTPUT_DIR}hot_pot_man_reviews.json", orient='split', indent=4)
aggregate_df

sentiment,Negative,Neutral,Positive
Taste,13.0,8.0,29.0
Portion,6.0,7.0,17.0
Price,15.0,5.0,13.0
Quality,16.0,11.0,35.0
Service,31.0,10.0,33.0
Speed,2.0,2.0,3.0
Location,13.0,8.0,22.0


---

# STEP 4: Priority Matrix

In [142]:
weights = {
	'Taste': 0.30,
	'Quality': 0.25,
	'Price': 0.15,
	'Service': 0.10,
	'Speed': 0.10,
	'Location': 0.05,
	'Portion': 0.05
}

In [143]:
def calculate_aspect_score(row):
	"""
	Converts Neg/Neu/Pos counts into a 0-10 score.
	Pos = 10, Neu = 5, Neg = 0
	"""
	total_mentions = row['Positive'] + row['Neutral'] + row['Negative']
	
	if total_mentions == 0:
		return 0.0 # Avoid division by zero if no mentions
	
	# Weighted sum of sentiments
	score_sum = (row['Positive'] * 10) + (row['Neutral'] * 5) + (row['Negative'] * 0)
	
	return score_sum / total_mentions

In [169]:
# 3. Apply the function to get a score (0-10) for each row
aggregate_df['Aspect_Score'] = aggregate_df.apply(calculate_aspect_score, axis=1)

# 4. Map the importance weights to the dataframe
aggregate_df['Weight'] = aggregate_df.index.map(weights)

# 5. Calculate final contribution (Score * Weight)
aggregate_df['Weighted_Contribution'] = aggregate_df['Aspect_Score'] * aggregate_df['Weight']

# 6. Final Result
final_review_score = aggregate_df['Weighted_Contribution'].sum()

# --- DISPLAY ---
print(aggregate_df[['Aspect_Score', 'Weight', 'Weighted_Contribution']])
print("-" * 30)
print(f"Final Thai Buffet Score: {final_review_score:.2f} / 10")

sentiment  Aspect_Score  Weight  Weighted_Contribution
Taste          6.600000    0.30               1.980000
Portion        6.833333    0.05               0.341667
Price          4.696970    0.15               0.704545
Quality        6.532258    0.25               1.633065
Service        5.135135    0.10               0.513514
Speed          5.714286    0.10               0.571429
Location       6.046512    0.05               0.302326
------------------------------
Final Thai Buffet Score: 6.05 / 10


In [170]:
# aggregate_df.to_json(f"{OUTPUT_DIR}bbq_plaza_reviews.json", orient='records', indent=4, lines=True)
aggregate_df

sentiment,Negative,Neutral,Positive,Aspect_Score,Weight,Weighted_Contribution
Taste,13.0,8.0,29.0,6.600000,0.30,1.980000
Portion,6.0,7.0,17.0,6.833333,0.05,0.341667
Price,15.0,5.0,13.0,4.696970,0.15,0.704545
Quality,16.0,11.0,35.0,6.532258,0.25,1.633065
Service,31.0,10.0,33.0,5.135135,0.10,0.513514
Speed,2.0,2.0,3.0,5.714286,0.10,0.571429
Location,13.0,8.0,22.0,6.046512,0.05,0.302326


In [171]:
overall_df = pd.DataFrame(aggregate_df.sum()).transpose().copy()
overall_df = overall_df.rename(columns={
	"Weighted_Contribution":"Score"
})
overall_df.index.name = None
overall_df[['Negative', "Neutral", "Positive", "Score"]]
overall_df['Name'] = "Hot pot man Buffet"
move_column = overall_df.pop('Name')
overall_df.insert(0, "Name", move_column)
overall_df

sentiment,Name,Negative,Neutral,Positive,Aspect_Score,Weight,Score
0,Hot pot man Buffet,96.0,51.0,152.0,41.558494,1.0,6.046544


In [172]:
shop_df = mock_review_df['sentiment']


shop_df = pd.DataFrame(shop_df.value_counts()).transpose()
shop_df['Name'] = "BBQ-Plaza"
move_column = shop_df.pop('Name')
shop_df.insert(0, "Name", move_column)

shop_df['Score'] = overall_df['Score'][0]
shop_df.reset_index()

sentiment,index,Name,Negative,Positive,Neutral,Score
0,count,BBQ-Plaza,43,41,16,6.046544


In [173]:
shop_df.to_json(f"{OUTPUT_DIR}test_reviews.json", orient='records', indent=4, lines=True)

---

# STEP 5: Text Generation

In [ ]:
# !pip install -q google-generativeai

import google.generativeai as genai
import os

# Setup API Key
genai.configure(api_key=MY_GEMINI_KEY)

# Prepare data string (same as above)
scores_text = ""
for aspect, row in aggregate_df.iterrows():
    scores_text += f"- {aspect}: {row['Aspect_Score']:.2f}/10 (Weight: {row['Weight']})\n"

prompt = f"""
Write a summary report in Thai for a restaurant management team based on these sentiment scores:
{scores_text}
Total Score: {final_review_score:.2f}/10

Please analyze:
1. The biggest problem areas (Lowest scores relative to weight)
2. What customers like (Highest scores)
3. Actionable recommendations
"""

# Call Model
model = genai.GenerativeModel('gemini-2.5-flash')
response = model.generate_content(prompt)

print(response.text)

เรียน ทีมผู้บริหารร้านอาหาร,

**รายงานสรุปผลคะแนนความพึงพอใจลูกค้า**

จากการวิเคราะห์คะแนนความพึงพอใจของลูกค้าในด้านต่างๆ พบว่าคะแนนภาพรวมของร้านอยู่ที่ **6.05/10** ซึ่งเป็นระดับที่พอใช้ได้ แต่ยังมีหลายประเด็นที่ต้องได้รับการปรับปรุงอย่างเร่งด่วนเพื่อยกระดับความพึงพอใจโดยรวมและส่งผลดีต่อธุรกิจ

---

**1. ประเด็นที่ต้องปรับปรุงเร่งด่วน (Biggest Problem Areas)**

เมื่อพิจารณาทั้งคะแนนและน้ำหนักความสำคัญ พบว่ามี 3 ประเด็นหลักที่จำเป็นต้องได้รับการแก้ไขอย่างจริงจัง:

*   **ราคา (Price): 4.70/10 (น้ำหนัก 0.15)**
    *   เป็นประเด็นที่มีคะแนนต่ำที่สุดอย่างเห็นได้ชัด และมีน้ำหนักความสำคัญค่อนข้างสูง ซึ่งบ่งชี้ว่าลูกค้ารู้สึกว่าราคาอาหารและบริการไม่คุ้มค่าหรือสูงเกินไปเมื่อเทียบกับสิ่งที่ได้รับ นี่คือปัญหาอันดับหนึ่งที่ส่งผลกระทบต่อภาพรวมความพึงพอใจอย่างมาก
*   **การบริการ (Service): 5.14/10 (น้ำหนัก 0.1)**
    *   คะแนนที่ต่ำในด้านการบริการแสดงให้เห็นถึงความไม่พึงพอใจในการปฏิสัมพันธ์กับพนักงาน ความเอาใจใส่ หรือมาตรฐานการบริการโดยรวม
*   **ความรวดเร็ว (Speed): 5.71/10 (น้ำหนัก 0.1)**
    *   ล